In [2]:
import os
import string
import pandas as pd
import pickle
import spacy
import random

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from nltk.tag import pos_tag
from nltk.classify import NaiveBayesClassifier, accuracy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import clear_output

CSV_PATH = "dataset/socialmedia.csv"
MODEL_PATH = "model/model_sosmed.pickle"

/opt/homebrew/Caskroom/miniforge/base/envs/ai_core/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
def get_tag(tag):
    if tag == 'jj':
        return 'a'
    elif tag in ['vb','nn','rb']:
        return tag[0]
    else:
        return 'n'

def lemmatizing(word_list):
    lemmatizer = WordNetLemmatizer()
    lemma_list = []
    tagged = pos_tag(word_list)
    for word, tag in tagged:
        label = get_tag(tag.lower())
        if label:
            result = lemmatizer.lemmatize(word, label)
            lemma_list.append(result)
        else:
            result = lemmatizer.lemmatize(word)
            lemma_list.append(result)
    return lemma_list


def preprocessing(sentence):
    eng_stopwords = stopwords.words('english')
    punctuation = string.punctuation

    word_list = word_tokenize(sentence.lower())
    remove_sw = [word for word in word_list if word not in eng_stopwords]
    remove_punc = [word for word in remove_sw if word not in punctuation]
    lemmatized = lemmatizing(remove_punc)

    return lemmatized


In [4]:
df = pd.read_csv(CSV_PATH)
print(df.info())
print(df.describe())

X = df['Reviews']
df['Label'] = df['Label'].map({'good': 'positive', 'neutral': 'neutral', 'not good': 'negative'})
y = df['Label']

all_sentences = ' '.join(X)
all_token = preprocessing(all_sentences)

freq_dist = FreqDist(all_token)

print(freq_dist.most_common(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 627 entries, 0 to 626
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Social Media Name  627 non-null    object
 1   Reviews            627 non-null    object
 2   Label              627 non-null    object
dtypes: object(3)
memory usage: 14.8+ KB
None
       Social Media Name   Reviews Label
count                627       627   627
unique                 7       595     3
top            Instagram  Good app  good
freq                 103         7   353
[('app', 230), ('instagram', 121), ("n't", 111), ("'s", 109), ('good', 104), ('please', 83), ('account', 68), ('post', 58), ('love', 47), ('like', 45)]


In [5]:
def extract_features(text):
    features = {}
    for word in freq_dist.keys():
        features[word] = (word in text)
    return features

features_sets = []
for Reviews, Label in zip(X, y):
    features = extract_features(preprocessing(Reviews))
    features_sets.append((features, Label))
    
print(features_sets[0])

({'facing': True, 'problem': True, 'whil': True, 'loging': True, 'temporarily': True, 'disabled': True, 'account': True, '2': True, 'day': True, 'ago': True, 'try': True, 'show': True, 'request': True, 'instagram': True, 'solve': True, 'good': False, 'use': False, 'update': False, 'love': False, 'app': False, 'picked': False, 'scammer': False, 'careful': False, 'recently': False, 'please': False, 'fix': False, 'thing': False, 'say': False, 'photo': False, 'posted': False, 'wo': False, "n't": False, 'go': False, 'away': False, "'m": False, 'surely': False, 'lending': False, 'phone': False, 'lock': False, 'cool': False, 'chat': False, 'like': False, 'omg': False, 'follow': False, 'support': False, '_ya__d_a_v': False, 'jyothi': False, 'beautiful': False, 'experiencing': False, 'crash': False, 'lately': False, '...': False, 'usually': False, 'great': False, 'help': False, 'abnormal': False, 'premix': False, 'baby': False, 'jaanu': False, 'mom': False, 'going': False, 'pls': False, 'even':

In [6]:
random.shuffle(features_sets)

train_count = int(len(features_sets) * 0.8)
train_set = features_sets[:train_count]
test_set = features_sets[train_count:]

In [7]:
def train_model():
    classifier = NaiveBayesClassifier.train(train_set)
    test_acc = accuracy(classifier, test_set)

    print("INFO MODEL: ")
    print(f"Accuracy: {test_acc*100}%")

    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(classifier,f)

    return classifier

In [11]:
def analyze_statement(statement, classifier):
    preprocessed_text = preprocessing(statement)
    extracted_text = extract_features(preprocessed_text)
    prediction = classifier.classify(extracted_text)
    return prediction

def write_statement():
    while True:
        statement = input("Write your Statement")
        if(len(statement) < 20):
            print("Invalid Input. Write atleas 20 characters")
        elif(len(statement.split()) < 3):
            print("Invalid Input. Write atleast 3 words")
        else:
            return statement
        
def tfidf_recommendation(curr_text):
    vectorizer = TfidfVectorizer(stop_words='english')
    corpus = df['Reviews']
    tfidf_matrix = vectorizer.fit_transform(corpus)

    user_vec = vectorizer.transform([' '.join(word_tokenize(curr_text.lower()))])
    similarity_scores = cosine_similarity(user_vec, tfidf_matrix).flatten()
    top_similar = similarity_scores.argsort()[-6:-1][::-1]

    print("\n Recommendation: ")
    for i, idx in enumerate(top_similar, 1):
        print(f"{i}.{df.iloc[idx]['Social Media Name']}")

def ner_text(curr_text):
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(curr_text)
    categories = {}

    for ent in doc.ents:
        label = ent.label_
        if label not in categories:
            categories[label] = []
        categories[label].append(ent.text)

    
    print("\n NAMED ENTITY RECOGNITION")
    for label, entities in categories.items():
        print(f'{label}: {", ".join(entities)}')

In [12]:
curr_text = None
curr_category = None
classifier = None

while True:
    clear_output(True)
    clear_output(True)
    print("SOSMED RAWR")
    print("your text: ", curr_text)
    print("your text category: ", curr_category)
    print("1. WRITE YOUR TEXT")
    print("2. View Job Posting Recommendation Based on Text")
    print("3. View NER")
    print("4. Exit")
    choice = input(">> ")
    if choice == '1':
        curr_text = write_statement()

        if os.path.exists(MODEL_PATH):
            with open(MODEL_PATH, "rb") as f:
                classifier = pickle.load(f)
        else:
            print("Model not found. Training Model...\n")
            classifier = train_model()

        curr_category = analyze_statement(curr_text, classifier)
        print("\nYour text: ", curr_text)
        print("Your Category: ", curr_category)
        input("\n Press enter to continue")
        continue

    elif choice == '2':
        if curr_text == None:
            print("\nNo Text. Please Write First!")
            input("\nPress enter to continue")
        
        tfidf_recommendation(curr_text)
        input("\n Press enter to continue")
        continue

    elif choice == '3':
        if curr_text == None:
            print("\nNo Text. Please Write First!")
            input("\nPress enter to continue")
            
        ner_text(curr_text)
        input("\n Press enter to continue")
        continue

    elif choice == '4':
        clear_output()
        print("\nExiting...")
        break

    else:
        print("\n\n PLEASE INPUT BETWEEN (1-4)!!!")
        input("\n Press enter to continue")
        continue



Exiting...
